# LTDS Network Planning with ukpyn

This tutorial covers the **Long Term Development Statement (LTDS)** orchestrator for accessing network planning data from UK Power Networks.

**What you'll learn:**

1. What is LTDS and why it's useful
2. Using the LTDS orchestrator with simple imports
3. Listing available datasets
4. Getting Table 3a (observed peak demand)
5. Getting Table 2a/2b (transformer data)
6. Getting Table 5 (generation data)
7. Getting Table 6 (connection interest)
8. Getting infrastructure projects
9. Exporting data to CSV

**Prerequisites:** Complete [01-getting-started.ipynb](01-getting-started.ipynb) first.

## 1. What is LTDS?

The **Long Term Development Statement (LTDS)** is a key regulatory document published by UK Power Networks twice yearly. It provides detailed information about the distribution network that is essential for:

- **Network planners** understanding infrastructure capacity
- **Developers** assessing connection opportunities
- **Researchers** analysing grid constraints and generation patterns
- **Local authorities** planning decarbonisation initiatives

### LTDS Tables Overview

| Table | Description | Key Use Cases |
|-------|-------------|---------------|
| **Table 2a/2b** | Transformer specifications | Capacity planning, asset modelling |
| **Table 3a** | Observed peak demand | Load forecasting, headroom analysis |
| **Table 5** | Generation capacity | Renewable energy mapping, curtailment analysis |
| **Table 6** | Connection interest | Development pipeline, constraint identification |
| **Projects** | Infrastructure schemes | Investment planning, reinforcement tracking |

### Licence Areas

UK Power Networks operates across three licence areas:

- **EPN** - Eastern Power Networks (East of England)
- **LPN** - London Power Networks (Greater London)
- **SPN** - South Eastern Power Networks (South East England)

## 2. Using the LTDS Orchestrator

The LTDS orchestrator provides a simple, ergonomic interface for accessing LTDS data. Import it directly from ukpyn:

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# Verify API key is set
if os.getenv("UKPN_API_KEY"):
    print("API key configured!")
else:
    print("Warning: UKPN_API_KEY not set. Set it to access the API.")

In [ ]:
# Import the LTDS orchestrator
from ukpyn import ltds

print("LTDS orchestrator imported successfully!")
print(f"Type: {type(ltds)}")

# Expected output:
# LTDS orchestrator imported successfully!
# Type: <module 'ukpyn.orchestrators.ltds' from '...'>

## 3. Listing Available Datasets

The `ltds.available_datasets` property shows all datasets you can access through the orchestrator.

In [ ]:
# List all available LTDS datasets
print("Available LTDS datasets:")
print("-" * 40)

for dataset in ltds.available_datasets:
    print(f"  - {dataset}")

print(f"\nTotal: {len(ltds.available_datasets)} datasets")

# Expected output:
# Available LTDS datasets:
# ----------------------------------------
#   - table_2a
#   - transformer_2w
#   - table_2b
#   - transformer_3w
#   - table_3a
#   - observed_peak_demand
#   - table_3a_transposed
#   - table_5
#   - generation
#   - table_6
#   - connection_interest
#   - infrastructure_projects
#   - projects
#   - cim
#
# Total: 14 datasets

### Using the Generic `get()` Function

The `ltds.get()` function allows you to fetch any LTDS dataset by name:

In [ ]:
# Fetch data using the generic get() function
response = ltds.get("table_3a", limit=5)

print(f"Dataset: table_3a")
print(f"Total records: {response.total_count}")
print(f"Records returned: {len(response.records)}")

# Show first record fields
if response.records:
    print(f"\nSample record fields:")
    for key, value in list(response.records[0].fields.items())[:5]:
        print(f"  {key}: {value}")

# Expected output:
# Dataset: table_3a
# Total records: 1523
# Records returned: 5
#
# Sample record fields:
#   licence_area: EPN
#   substation: ALDEBURGH PRIMARY
#   year: 2023
#   peak_demand_mw: 12.5
#   ...

## 4. Table 3a - Observed Peak Demand

Table 3a contains observed load data for primary substations. This is essential for:

- Understanding current network utilisation
- Identifying substations approaching capacity
- Load forecasting and trend analysis

Use the dedicated `get_table_3a()` function for convenient filtering:

In [ ]:
# Get Table 3a data for a specific licence area
epn_demand = ltds.get_table_3a(licence_area="EPN", limit=10)

print(f"EPN Observed Peak Demand Data")
print(f"Total records: {epn_demand.total_count}")
print(f"Records returned: {len(epn_demand.records)}")
print("-" * 60)

for record in epn_demand.records[:5]:
    fields = record.fields
    substation = fields.get("substation", "Unknown")
    peak = fields.get("peak_demand_mw", "N/A")
    year = fields.get("year", "N/A")
    print(f"  {substation}: {peak} MW (Year: {year})")

# Expected output:
# EPN Observed Peak Demand Data
# Total records: 523
# Records returned: 10
# ------------------------------------------------------------
#   ALDEBURGH PRIMARY: 12.5 MW (Year: 2023)
#   ALTON WATER PRIMARY: 8.3 MW (Year: 2023)
#   AMPTON PRIMARY: 15.2 MW (Year: 2023)
#   ...

In [ ]:
# Filter by licence area and year
lpn_2023 = ltds.get_table_3a(licence_area="LPN", year=2023, limit=10)

print(f"London Power Networks - 2023 Peak Demand")
print(f"Total records: {lpn_2023.total_count}")
print("-" * 60)

for record in lpn_2023.records[:5]:
    fields = record.fields
    print(f"  {fields.get('substation', 'Unknown')}: {fields.get('peak_demand_mw', 'N/A')} MW")

# Expected output:
# London Power Networks - 2023 Peak Demand
# Total records: 187
# ------------------------------------------------------------
#   ABBEY ROAD PRIMARY: 22.1 MW
#   ACTON LANE PRIMARY: 18.7 MW
#   ...

## 5. Table 2a/2b - Transformer Data

Tables 2a and 2b contain transformer specifications:

- **Table 2a**: Two-winding transformers (1x HV, 1x LV)
- **Table 2b**: Three-winding transformers (1x HV, 2x LV)

This data is useful for capacity planning and network modelling.

In [ ]:
# Get Table 2a - Two-winding transformer data
transformers_2w = ltds.get_table_2a(licence_area="SPN", limit=10)

print(f"SPN Two-Winding Transformers (Table 2a)")
print(f"Total records: {transformers_2w.total_count}")
print("-" * 60)

for record in transformers_2w.records[:5]:
    fields = record.fields
    substation = fields.get("substation", "Unknown")
    rating = fields.get("rating_mva", "N/A")
    voltage_hv = fields.get("voltage_hv_kv", "N/A")
    voltage_lv = fields.get("voltage_lv_kv", "N/A")
    print(f"  {substation}: {rating} MVA ({voltage_hv}/{voltage_lv} kV)")

# Expected output:
# SPN Two-Winding Transformers (Table 2a)
# Total records: 234
# ------------------------------------------------------------
#   ASHFORD GRID: 90 MVA (132/33 kV)
#   BEXHILL PRIMARY: 20 MVA (33/11 kV)
#   ...

In [ ]:
# Get Table 2b - Three-winding transformer data
transformers_3w = ltds.get_table_2b(licence_area="EPN", limit=10)

print(f"EPN Three-Winding Transformers (Table 2b)")
print(f"Total records: {transformers_3w.total_count}")
print("-" * 60)

for record in transformers_3w.records[:5]:
    fields = record.fields
    substation = fields.get("substation", "Unknown")
    rating = fields.get("rating_mva", "N/A")
    print(f"  {substation}: {rating} MVA")

# Expected output:
# EPN Three-Winding Transformers (Table 2b)
# Total records: 45
# ------------------------------------------------------------
#   BRAMFORD GRID: 240 MVA
#   BURWELL GRID: 180 MVA
#   ...

In [ ]:
# Search for a specific substation by name
search_results = ltds.get_table_2a(substation="ASHFORD", limit=20)

print(f"Transformers matching 'ASHFORD'")
print(f"Found: {search_results.total_count} records")
print("-" * 60)

for record in search_results.records:
    fields = record.fields
    print(f"  [{fields.get('licence_area')}] {fields.get('substation')}: {fields.get('rating_mva')} MVA")

# Expected output:
# Transformers matching 'ASHFORD'
# Found: 3 records
# ------------------------------------------------------------
#   [SPN] ASHFORD GRID: 90 MVA
#   [SPN] ASHFORD PRIMARY: 20 MVA
#   ...

## 6. Table 5 - Generation Data

Table 5 shows the capacity of existing distributed generation connected at each primary substation. This includes:

- Solar PV installations
- Wind generation
- Battery storage
- Other distributed energy resources

Essential for understanding renewable energy penetration and export constraints.

In [ ]:
# Get generation data for a licence area
generation = ltds.get_table_5(licence_area="EPN", limit=10)

print(f"EPN Generation Capacity (Table 5)")
print(f"Total records: {generation.total_count}")
print("-" * 60)

for record in generation.records[:5]:
    fields = record.fields
    substation = fields.get("substation", "Unknown")
    tech = fields.get("technology_type", "Unknown")
    capacity = fields.get("installed_capacity_mw", "N/A")
    print(f"  {substation} ({tech}): {capacity} MW")

# Expected output:
# EPN Generation Capacity (Table 5)
# Total records: 1247
# ------------------------------------------------------------
#   ALDEBURGH PRIMARY (Solar): 5.2 MW
#   ALTON WATER PRIMARY (Wind): 12.0 MW
#   ...

In [ ]:
# Filter by technology type - Solar generation
solar = ltds.get_table_5(technology_type="Solar", limit=10)

print(f"Solar Generation Across All Licence Areas")
print(f"Total records: {solar.total_count}")
print("-" * 60)

for record in solar.records[:5]:
    fields = record.fields
    area = fields.get("licence_area", "?")
    substation = fields.get("substation", "Unknown")
    capacity = fields.get("installed_capacity_mw", "N/A")
    print(f"  [{area}] {substation}: {capacity} MW")

# Expected output:
# Solar Generation Across All Licence Areas
# Total records: 892
# ------------------------------------------------------------
#   [EPN] ATTLEBOROUGH PRIMARY: 8.5 MW
#   [SPN] BATTLE PRIMARY: 3.2 MW
#   ...

In [ ]:
# Combine filters: Wind generation in SPN
spn_wind = ltds.get_table_5(licence_area="SPN", technology_type="Wind", limit=10)

print(f"Wind Generation in South Eastern Power Networks")
print(f"Total records: {spn_wind.total_count}")
print("-" * 60)

for record in spn_wind.records:
    fields = record.fields
    print(f"  {fields.get('substation')}: {fields.get('installed_capacity_mw')} MW")

# Expected output:
# Wind Generation in South Eastern Power Networks
# Total records: 23
# ------------------------------------------------------------
#   DUNGENESS PRIMARY: 25.0 MW
#   RYE PRIMARY: 8.5 MW
#   ...

## 7. Table 6 - Connection Interest

Table 6 indicates the level of new connection interest at each primary substation. This shows the pipeline of:

- Queued generation connections
- Queued demand connections
- Accepted but not yet connected projects

Critical for understanding future network constraints and development hotspots.

In [ ]:
# Get connection interest data
connections = ltds.get_table_6(licence_area="LPN", limit=10)

print(f"London Power Networks - Connection Interest (Table 6)")
print(f"Total records: {connections.total_count}")
print("-" * 60)

for record in connections.records[:5]:
    fields = record.fields
    substation = fields.get("substation", "Unknown")
    gen_queue = fields.get("generation_queue_mw", "N/A")
    demand_queue = fields.get("demand_queue_mw", "N/A")
    print(f"  {substation}:")
    print(f"    Generation queue: {gen_queue} MW")
    print(f"    Demand queue: {demand_queue} MW")

# Expected output:
# London Power Networks - Connection Interest (Table 6)
# Total records: 187
# ------------------------------------------------------------
#   ABBEY ROAD PRIMARY:
#     Generation queue: 2.5 MW
#     Demand queue: 15.0 MW
#   ...

In [ ]:
# Search for connection interest at specific substations
battersea = ltds.get_table_6(substation="BATTERSEA", limit=10)

print(f"Connection Interest - Substations matching 'BATTERSEA'")
print(f"Found: {battersea.total_count} records")
print("-" * 60)

for record in battersea.records:
    fields = record.fields
    print(f"\n  {fields.get('substation')}")
    for key, value in fields.items():
        if key != "substation" and value is not None:
            print(f"    {key}: {value}")

# Expected output:
# Connection Interest - Substations matching 'BATTERSEA'
# Found: 2 records
# ------------------------------------------------------------
#   BATTERSEA PRIMARY
#     licence_area: LPN
#     generation_queue_mw: 5.0
#     demand_queue_mw: 25.0
#   ...

## 8. Infrastructure Projects

The infrastructure projects dataset contains information about UK Power Networks' network investment and reinforcement schemes, including:

- Network reinforcement projects
- New connection schemes
- Asset replacement programmes
- Innovation projects

In [ ]:
# Get infrastructure projects
projects = ltds.get_projects(limit=10)

print(f"UKPN Infrastructure Projects")
print(f"Total records: {projects.total_count}")
print("-" * 60)

for record in projects.records[:5]:
    fields = record.fields
    name = fields.get("project_name", "Unknown")
    status = fields.get("status", "N/A")
    ptype = fields.get("project_type", "N/A")
    print(f"  {name}")
    print(f"    Type: {ptype}")
    print(f"    Status: {status}")
    print()

# Expected output:
# UKPN Infrastructure Projects
# Total records: 342
# ------------------------------------------------------------
#   Bramford-Twinstead Reinforcement
#     Type: Reinforcement
#     Status: Active
#
#   Canterbury Grid Upgrade
#     Type: Asset Replacement
#     Status: Planning
#   ...

In [ ]:
# Filter projects by status
active_projects = ltds.get_projects(status="Active", limit=10)

print(f"Active Infrastructure Projects")
print(f"Total: {active_projects.total_count}")
print("-" * 60)

for record in active_projects.records[:5]:
    fields = record.fields
    name = fields.get("project_name", "Unknown")
    completion = fields.get("expected_completion", "TBD")
    print(f"  {name} (Expected: {completion})")

# Expected output:
# Active Infrastructure Projects
# Total: 78
# ------------------------------------------------------------
#   Bramford-Twinstead Reinforcement (Expected: 2026)
#   Isle of Grain Connection (Expected: 2025)
#   ...

## 9. Exporting Data to CSV

The LTDS orchestrator includes an `export()` function for downloading data in various formats including CSV, JSON, and Excel.

In [ ]:
# Export Table 3a to CSV
from pathlib import Path

csv_data = ltds.export("table_3a", format="csv", limit=100)

# Save to file only when save_dir is set
save_dir = None  # Set to a directory (e.g. "exports") to enable writing files.
if save_dir:
    output_file = Path(save_dir) / "ltds_table_3a.csv"
    output_file.parent.mkdir(parents=True, exist_ok=True)
    with open(output_file, "wb") as f:
        f.write(csv_data)
    print(f"Exported {len(csv_data)} bytes to {output_file}")
else:
    print(f"Exported {len(csv_data)} bytes (file save skipped; set save_dir to enable writing).")

# Preview first few lines
print("\nPreview (first 500 characters):")
print(csv_data.decode("utf-8")[:500])

# Expected output:
# Exported 12453 bytes
#
# Preview (first 500 characters):
# licence_area;substation;year;peak_demand_mw;...
# EPN;ALDEBURGH PRIMARY;2023;12.5;...
# ...

In [ ]:
# Export generation data to CSV
from pathlib import Path

gen_csv = ltds.export("table_5", format="csv", limit=200)

save_dir = None  # Set to a directory (e.g. "exports") to enable writing files.
if save_dir:
    output_file = Path(save_dir) / "ltds_generation.csv"
    output_file.parent.mkdir(parents=True, exist_ok=True)
    with open(output_file, "wb") as f:
        f.write(gen_csv)
    print(f"Exported generation data: {len(gen_csv)} bytes to {output_file}")
else:
    print(f"Exported generation data: {len(gen_csv)} bytes (file save skipped; set save_dir to enable writing).")

# Expected output:
# Exported generation data: 24891 bytes

In [ ]:
# Export to JSON format
import json

json_data = ltds.export("table_6", format="json", limit=10)

# Parse and pretty-print
data = json.loads(json_data)
print(f"Exported {len(data)} records as JSON")
print("\nFirst record:")
print(json.dumps(data[0], indent=2))

# Expected output:
# Exported 10 records as JSON
#
# First record:
# {
#   "licence_area": "EPN",
#   "substation": "ALDEBURGH PRIMARY",
#   "generation_queue_mw": 1.5,
#   ...
# }

### Loading CSV into pandas

In [ ]:
# Load exported CSV into pandas for analysis
try:
    import pandas as pd
    from io import BytesIO
    
    # Export and load directly
    csv_bytes = ltds.export("table_3a", format="csv", limit=500)
    df = pd.read_csv(BytesIO(csv_bytes), sep=";")
    
    print(f"Loaded DataFrame: {df.shape[0]} rows x {df.shape[1]} columns")
    print(f"\nColumns: {list(df.columns)}")
    print(f"\nFirst 5 rows:")
    display(df.head())
    
except ImportError:
    print("pandas not installed. Install with: pip install pandas")

# Expected output:
# Loaded DataFrame: 500 rows x 12 columns
#
# Columns: ['licence_area', 'substation', 'year', 'peak_demand_mw', ...]
#
# First 5 rows:
#   licence_area          substation  year  peak_demand_mw  ...
# 0          EPN  ALDEBURGH PRIMARY  2023            12.5  ...
# ...

## Working with Async (Advanced)

For Jupyter notebooks or async applications, you can use the async versions of the functions:

In [ ]:
# Async version - works natively in Jupyter
response = await ltds.get_async("table_3a", limit=5)

print(f"Async fetch complete!")
print(f"Records: {len(response.records)}")

# Expected output:
# Async fetch complete!
# Records: 5

## Summary

You've learned how to:

- **Import the LTDS orchestrator** with `from ukpyn import ltds`
- **List available datasets** using `ltds.available_datasets`
- **Fetch data** using the generic `ltds.get()` or table-specific functions
- **Filter by licence area** (EPN, LPN, SPN) and other parameters
- **Access Table 3a** for peak demand data
- **Access Tables 2a/2b** for transformer specifications
- **Access Table 5** for generation capacity by technology
- **Access Table 6** for connection interest pipelines
- **Get infrastructure projects** and filter by status
- **Export data** to CSV, JSON, and other formats

## Function Reference

| Function | Description |
|----------|-------------|
| `ltds.get(dataset, ...)` | Generic fetch for any LTDS dataset |
| `ltds.get_table_2a(...)` | Transformer data (2-winding) |
| `ltds.get_table_2b(...)` | Transformer data (3-winding) |
| `ltds.get_table_3a(...)` | Observed peak demand |
| `ltds.get_table_5(...)` | Generation capacity |
| `ltds.get_table_6(...)` | Connection interest |
| `ltds.get_projects(...)` | Infrastructure projects |
| `ltds.get_cim(...)` | Common Information Model data |
| `ltds.export(...)` | Export data to CSV/JSON/Excel |

## Next Steps

- Explore the [03-analysis-patterns.ipynb](03-analysis-patterns.ipynb) for data analysis with pandas
- Check out the [examples](../examples/) folder for community contributions
- Visit the [UK Power Networks Open Data Portal](https://ukpowernetworks.opendatasoft.com/) for more datasets